# Parquet Basics — 07: Small File Problem

## What you will learn
The small file problem is the most common performance issue in data lakes.
Thousands of tiny Parquet files cause slow query planning, high metadata
overhead, and poor I/O throughput.

In this notebook:
1. What causes the small file problem
2. Measuring the impact on query performance
3. Solutions: coalesce, repartition, AQE coalescing
4. Compaction — merging small files after the fact
5. Preventing small files — write-time strategies
6. Monitoring file count and size distribution


In [1]:
import os, time, pathlib, shutil, random, datetime, subprocess
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

MASTER   = 'spark://spark-master:7077'
DATA_DIR = '/workspace/data/parquet_basics'
pathlib.Path(DATA_DIR).mkdir(parents=True, exist_ok=True)

spark = (
    SparkSession.builder
    .appName('parquet-basics')
    .master(MASTER)
    .config('spark.executor.memory', '2g')
    .config('spark.driver.memory',   '1g')
    .config('spark.executor.cores',  '2')
    .config('spark.shuffle.sort.bypassMergeThreshold', '0')
    .getOrCreate()
)
spark.sparkContext.setLogLevel('WARN')
print(f"Spark {spark.version} | DATA_DIR: {DATA_DIR}")

random.seed(42)
N = 200_000
CATEGORIES = ["Electronics","Clothing","Books","Food","Sports","Furniture"]
COUNTRIES  = ["US","UK","DE","FR","JP","CA","AU","BR"]
STATUSES   = ["pending","confirmed","shipped","delivered","cancelled"]

schema = StructType([
    StructField("order_id",    LongType()),
    StructField("customer_id", LongType()),
    StructField("product",     StringType()),
    StructField("category",    StringType()),
    StructField("country",     StringType()),
    StructField("quantity",    IntegerType()),
    StructField("price",       DoubleType()),
    StructField("discount",    DoubleType()),
    StructField("revenue",     DoubleType()),
    StructField("status",      StringType()),
    StructField("is_returned", BooleanType()),
    StructField("order_date",  DateType()),
])

base = datetime.date(2023, 1, 1)
rows = []
for i in range(N):
    qty  = random.randint(1, 20)
    price = round(random.uniform(5, 2000), 2)
    disc  = round(random.uniform(0, 0.4), 3)
    rev   = round(qty * price * (1 - disc), 2)
    d     = base + datetime.timedelta(days=random.randint(0, 365))
    rows.append((i+1, random.randint(1,50000),
                 f"Product_{random.randint(1,500)}",
                 random.choice(CATEGORIES), random.choice(COUNTRIES),
                 qty, price, disc, rev,
                 random.choice(STATUSES), random.random() < 0.05, d))

df = spark.createDataFrame(rows, schema)
df.cache().count()
print(f"Dataset: {N:,} rows | {len(schema)} columns")
df.printSchema()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/10 21:17:58 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark 4.0.2 | DATA_DIR: /workspace/data/parquet_basics


26/04/10 21:18:07 WARN TaskSetManager: Stage 0 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:18:09 WARN TaskSetManager: Stage 1 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.


Dataset: 200,000 rows | 12 columns
root
 |-- order_id: long (nullable = true)
 |-- customer_id: long (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- country: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- discount: double (nullable = true)
 |-- revenue: double (nullable = true)
 |-- status: string (nullable = true)
 |-- is_returned: boolean (nullable = true)
 |-- order_date: date (nullable = true)



## Step 1 — Creating the Small File Problem

Small files are created when you write many small batches, have
too many shuffle partitions, or partition by high-cardinality columns.


In [2]:
SMALL_PATH  = f"{DATA_DIR}/small_files"
LARGE_PATH  = f"{DATA_DIR}/large_files"

# Create small file problem: 200 tiny files
df.repartition(200).write.mode("overwrite").parquet(SMALL_PATH)

# Equivalent data in few large files
df.coalesce(4).write.mode("overwrite").parquet(LARGE_PATH)

import glob as glib

small_files = glib.glob(f"{SMALL_PATH}/*.parquet")
large_files = glib.glob(f"{LARGE_PATH}/*.parquet")

small_sizes = [pathlib.Path(f).stat().st_size for f in small_files]
large_sizes = [pathlib.Path(f).stat().st_size for f in large_files]

print(f"Small files setup:")
print(f"  File count : {len(small_files)}")
print(f"  Avg size   : {sum(small_sizes)/len(small_sizes)/1024:.1f} KB per file")
print(f"  Total size : {sum(small_sizes)/1024/1024:.1f} MB")
print()
print(f"Large files setup:")
print(f"  File count : {len(large_files)}")
print(f"  Avg size   : {sum(large_sizes)/len(large_sizes)/1024:.1f} KB per file")
print(f"  Total size : {sum(large_sizes)/1024/1024:.1f} MB")

26/04/10 21:18:10 WARN TaskSetManager: Stage 4 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:18:31 WARN TaskSetManager: Stage 7 contains a task of very large size (3058 KiB). The maximum recommended task size is 1000 KiB.


Small files setup:
  File count : 200
  Avg size   : 33.5 KB per file
  Total size : 6.5 MB

Large files setup:
  File count : 4
  Avg size   : 1183.1 KB per file
  Total size : 4.6 MB


## Step 2 — Measuring the Performance Impact


In [3]:
queries = [
    ("Full scan + agg",
     lambda p: spark.read.parquet(p).agg(F.sum("revenue"), F.count("*")).collect()),
    ("Filter + agg",
     lambda p: spark.read.parquet(p)
               .filter(F.col("category") == "Electronics")
               .agg(F.sum("revenue")).collect()),
    ("GroupBy + agg",
     lambda p: spark.read.parquet(p)
               .groupBy("category").agg(F.sum("revenue")).collect()),
]

print(f"{'Query':<25} {'Small (200 files)':>18} {'Large (4 files)':>16} {'Speedup':>10}")
print("-" * 75)

for label, fn in queries:
    ts, tl = [], []
    for _ in range(3):
        t0=time.time(); fn(SMALL_PATH); ts.append(time.time()-t0)
        t0=time.time(); fn(LARGE_PATH); tl.append(time.time()-t0)
    t_s, t_l = sorted(ts)[1], sorted(tl)[1]
    print(f"  {label:<23} {t_s:>16.3f}s {t_l:>14.3f}s {t_l and t_s/t_l:>9.1f}x")

print()
print("The overhead comes from:")
print("  1. Opening 200 files instead of 4 (file system metadata)")
print("  2. Reading 200 Parquet footers instead of 4")
print("  3. Scheduling 200 tasks instead of 4 (Spark overhead)")

Query                      Small (200 files)  Large (4 files)    Speedup
---------------------------------------------------------------------------


  Full scan + agg                    5.406s          0.324s      16.7x


  Filter + agg                       5.419s          0.323s      16.8x


  GroupBy + agg                      5.304s          0.315s      16.8x

The overhead comes from:
  1. Opening 200 files instead of 4 (file system metadata)
  2. Reading 200 Parquet footers instead of 4
  3. Scheduling 200 tasks instead of 4 (Spark overhead)


## Step 3 — Fixing Small Files: coalesce vs repartition


In [4]:
# Fix 1: coalesce — reduce partitions without full shuffle (cheap)
t0 = time.time()
df.repartition(200).coalesce(4).write.mode("overwrite").parquet(f"{DATA_DIR}/fixed_coalesce")
t_coalesce = time.time() - t0

# Fix 2: repartition — full shuffle, even distribution (expensive but balanced)
t0 = time.time()
df.repartition(4).write.mode("overwrite").parquet(f"{DATA_DIR}/fixed_repartition")
t_repartition = time.time() - t0

for label, path, t_write in [
    ("coalesce(4)",     f"{DATA_DIR}/fixed_coalesce",     t_coalesce),
    ("repartition(4)",  f"{DATA_DIR}/fixed_repartition",  t_repartition),
]:
    files = glib.glob(f"{path}/*.parquet")
    sizes = [pathlib.Path(f).stat().st_size for f in files]
    avg_mb = sum(sizes)/len(sizes)/1024/1024
    print(f"{label}:")
    print(f"  Write time : {t_write:.2f}s")
    print(f"  Files      : {len(files)}")
    print(f"  Avg size   : {avg_mb:.1f} MB per file")
    sizes_mb = sorted([s/1024/1024 for s in sizes])
    print(f"  Size range : {min(sizes_mb):.1f} – {max(sizes_mb):.1f} MB")
    print()

print("coalesce    : cheap (no shuffle), but may produce uneven file sizes")
print("repartition : expensive (full shuffle), produces even file sizes")

26/04/10 21:19:25 WARN TaskSetManager: Stage 80 contains a task of very large size (3057 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:19:26 WARN TaskSetManager: Stage 83 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


coalesce(4):
  Write time : 1.04s
  Files      : 4
  Avg size   : 1.2 MB per file
  Size range : 1.2 – 1.2 MB

repartition(4):
  Write time : 0.84s
  Files      : 4
  Avg size   : 1.2 MB per file
  Size range : 1.2 – 1.2 MB

coalesce    : cheap (no shuffle), but may produce uneven file sizes
repartition : expensive (full shuffle), produces even file sizes


## Step 4 — Compaction: Merging Existing Small Files


In [5]:
# Compaction: read small files, write as fewer large files
COMPACTED_PATH = f"{DATA_DIR}/compacted"

print(f"Before compaction: {len(glib.glob(SMALL_PATH+'/*.parquet'))} files")

t0 = time.time()
(spark.read.parquet(SMALL_PATH)    # read all small files
      .repartition(4)              # consolidate to 4 partitions
      .write.mode("overwrite")
      .parquet(COMPACTED_PATH))    # write as 4 large files
t_compact = time.time() - t0

compacted_files = glib.glob(f"{COMPACTED_PATH}/*.parquet")
print(f"After compaction : {len(compacted_files)} files  ({t_compact:.2f}s)")

# Verify data integrity
original_count  = spark.read.parquet(SMALL_PATH).count()
compacted_count = spark.read.parquet(COMPACTED_PATH).count()
print(f"\nData integrity check:")
print(f"  Original  : {original_count:,} rows")
print(f"  Compacted : {compacted_count:,} rows")
print(f"  Match     : {'✅ PASS' if original_count == compacted_count else '❌ FAIL'}")

# Query benchmark after compaction
t0=time.time(); spark.read.parquet(SMALL_PATH).agg(F.sum("revenue")).collect(); t_before=time.time()-t0
t0=time.time(); spark.read.parquet(COMPACTED_PATH).agg(F.sum("revenue")).collect(); t_after=time.time()-t0
print(f"\nQuery speedup after compaction: {t_before/t_after:.1f}x")

Before compaction: 200 files


After compaction : 4 files  (6.24s)



Data integrity check:
  Original  : 200,000 rows
  Compacted : 200,000 rows
  Match     : ✅ PASS



Query speedup after compaction: 18.9x


## Step 5 — AQE Coalescing: Automatic Fix

AQE's partition coalescing automatically merges small post-shuffle partitions.
This is the most seamless solution for streaming and batch pipelines.


In [6]:
# Demonstrate AQE coalescing
spark.conf.set("spark.sql.adaptive.enabled", "true")
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.shuffle.partitions", "200")

# Without AQE — 200 tiny shuffle partitions
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "false")
t0 = time.time()
r1 = df.groupBy("category").agg(F.sum("revenue")).rdd.getNumPartitions()
t_no_aqe = time.time() - t0

# With AQE coalescing — merged to fewer partitions
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
t0 = time.time()
r2 = df.groupBy("category").agg(F.sum("revenue")).rdd.getNumPartitions()
t_aqe = time.time() - t0

print(f"Without AQE coalescing: {r1} output partitions  {t_no_aqe:.3f}s")
print(f"With AQE coalescing   : {r2} output partitions  {t_aqe:.3f}s")
print()
print("AQE merged tiny post-shuffle partitions automatically.")
print("No code changes needed — just enable AQE (default in Spark 3.0+)")

26/04/10 21:19:44 WARN TaskSetManager: Stage 106 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.
26/04/10 21:19:44 WARN TaskSetManager: Stage 107 contains a task of very large size (3126 KiB). The maximum recommended task size is 1000 KiB.


Without AQE coalescing: 200 output partitions  0.196s
With AQE coalescing   : 1 output partitions  0.122s

AQE merged tiny post-shuffle partitions automatically.
No code changes needed — just enable AQE (default in Spark 3.0+)


## Summary: Small File Problem Cheat Sheet

### Causes
- Many small batches written to same table
- `spark.sql.shuffle.partitions=200` (default) for small data
- Partitioning by high-cardinality column
- Streaming micro-batches writing many small files

### Solutions at write time
```python
# Reduce partitions before write
df.coalesce(4).write.parquet(path)         # cheap, may be uneven
df.repartition(4).write.parquet(path)      # even, needs shuffle

# For streaming — use trigger.AvailableNow or larger triggers
```

### Solutions after the fact (compaction)
```python
spark.read.parquet(path)
     .repartition(n_files)
     .write.mode("overwrite").parquet(path)
```

### Automatic solutions
- **AQE coalescing** — auto-merges small post-shuffle partitions
- **Delta OPTIMIZE** — built-in compaction
- **Iceberg rewrite_data_files** — built-in compaction

### File size targets
- Minimum: 64 MB per file
- Optimal: 128–512 MB per file
- Maximum: 1 GB per file (diminishing returns above this)
